# Load dependencies

In [ ]:
import pickle,gzip,torch
import matplotlib as mpl
from pathlib import Path
from torch import tensor,nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [ ]:
# from fastcore.test import test_close
# https://github.com/AnswerDotAI/fastcore/blob/main/fastcore/test.py
from functools import partial
from typing import Iterable,Generator
def is_close(a,b,eps=1e-5):
    "Is `a` within `eps` of `b`"
    if hasattr(a, '__array__') or hasattr(b,'__array__'):
        return (abs(a-b)<eps).all()
    if isinstance(a, (Iterable,Generator)) or isinstance(b, (Iterable,Generator)):
        return all(abs(a_-b_)<eps for a_,b_ in zip(a,b))
    return abs(a-b)<eps

def test(a, b, cmp, cname=None):
    "`assert` that `cmp(a,b)`; display inputs and `cname or cmp.__name__` if it fails"
    if cname is None: cname=cmp.__name__
    assert cmp(a,b),f"{cname}:\n{a}\n{b}"

def test_close(a,b,eps=1e-5):
    "`test` that `a` is within `eps` of `b`"
    test(a,b,partial(is_close,eps=eps),'close')

# Config

In [ ]:
torch.set_printoptions(precision=2, linewidth=140, sci_mode=False)
torch.manual_seed(1)
mpl.rcParams['image.cmap'] = 'gray'

# Load data

In [ ]:
path_data = Path('data')
path_gz = path_data/'mnist.pkl.gz'
with gzip.open(path_gz, 'rb') as f: ((x_train, y_train), (x_valid, y_valid), _) = pickle.load(f, encoding='latin-1')
x_train, y_train, x_valid, y_valid = map(tensor, [x_train, y_train, x_valid, y_valid])

# Define Model

In [ ]:
class Model(nn.Module):
    def __init__(self, n_in, nh, n_out):
        super().__init__()
        self.layers = [nn.Linear(n_in,nh), nn.ReLU(), nn.Linear(nh,n_out)]
        
    def __call__(self, x):
        for l in self.layers: x = l(x)
        return x

## Test a single call

In [ ]:
n,m = x_train.shape
c = y_train.max()+1
nh = 50

model = Model(m, nh, 10)
pred = model(x_train)
print(pred.shape)

# Define Loss

## Log softmax values

First, we will need to compute the softmax of our activations. This is defined by:

$$\hbox{softmax(x)}_{i} = \frac{e^{x_{i}}}{e^{x_{0}} + e^{x_{1}} + \cdots + e^{x_{n-1}}}$$

or more concisely:

$$\hbox{softmax(x)}_{i} = \frac{e^{x_{i}}}{\sum\limits_{0 \leq j \lt n} e^{x_{j}}}$$

In practice, we will need the log of the softmax when we calculate the loss.

Note that the formula 

$$\log \left ( \frac{a}{b} \right ) = \log(a) - \log(b)$$

gives a simplification when we compute the log softmax.

Then, there is a way to compute the log of the sum of exponentials in a more stable way, called the [LogSumExp trick](https://en.wikipedia.org/wiki/LogSumExp). The idea is to use the following formula:

$$\log \left ( \sum_{j=1}^{n} e^{x_{j}} \right ) = \log \left ( e^{a} \sum_{j=1}^{n} e^{x_{j}-a} \right ) = a + \log \left ( \sum_{j=1}^{n} e^{x_{j}-a} \right )$$

where a is the maximum of the $x_{j}$.

This way, we will avoid an overflow when taking the exponential of a big activation. In PyTorch, this is already implemented for us. 

In [ ]:
def logsumexp(x):
    m = x.max(-1)[0]
    return m + (x-m[:,None]).exp().sum(-1).log()

def log_softmax(x): return x - x.logsumexp(-1,keepdim=True)

# compare with the pytorch implementation
test_close(logsumexp(pred), pred.logsumexp(-1))

sm_pred = log_softmax(pred)
print(sm_pred) # these are log values of probabilities

## Cross entropy loss

The cross entropy loss for some target $x$ and some prediction $p(x)$ is given by:

$$ -\sum x\, \log p(x) $$

But since our $x$'s are 1-hot encoded (actually, they're just the integer indices), this can be rewritten as $-\log(p_{i})$ where i is the index of the desired target.

This can be done using numpy-style [integer array indexing](https://docs.scipy.org/doc/numpy-1.13.0/reference/arrays.indexing.html#integer-array-indexing). Note that PyTorch supports all the tricks in the advanced indexing methods discussed in that link.

In [ ]:
print(y_train[:3])
print(sm_pred[0,5],sm_pred[1,0],sm_pred[2,4])
print(sm_pred[range(3), y_train[:3]])

In [ ]:
def nll(input, target): return -input[range(target.shape[0]), target].mean()
loss = nll(sm_pred, y_train)
print(loss)

Compare with PyTorch's implementation.

In PyTorch, `F.log_softmax` and `F.nll_loss` are combined in one optimized function, `F.cross_entropy`.

In [ ]:
test_close(F.nll_loss(F.log_softmax(pred, -1), y_train), loss, 1e-3)
test_close(F.cross_entropy(pred, y_train), loss, 1e-3)

# Basic training loop

Basically the training loop repeats over the following steps:
- get the output of the model on a batch of inputs
- compare the output to the labels we have and compute a loss
- calculate the gradients of the loss with respect to every parameter of the model
- update said parameters with those gradients to make them a little bit better

In [ ]:
loss_func = F.cross_entropy

bs=50                  # batch size
xb = x_train[0:bs]     # a mini-batch from x
preds = model(xb)      # predictions
print(f"first record's logits: {preds[0].data}")
print(f"batch prediction shape: {preds.shape}")

yb = y_train[0:bs]
print(f"batch true labels: {yb}")
print(f"batch predicted labels: {preds.argmax(dim=1)}")

print(f"loss: {loss_func(preds, yb)}")

def accuracy(out, yb): return (out.argmax(dim=1)==yb).float().mean()
print(f"accuracy: {accuracy(preds, yb)}")

def report(loss, preds, yb): print(f'loss: {loss:.5f}, accuracy: {accuracy(preds, yb):.3f}')
xb,yb = x_train[:bs],y_train[:bs]
preds = model(xb)
report(loss_func(preds, yb), preds, yb)

In [ ]:
def eval(model, x, y):
    with torch.no_grad():
        preds = model(x)
        loss = loss_func(preds, y)
        report(loss, preds, y)

def fit_0(model, x_train, y_train, bs, lr = 0.5, epochs = 3):
    print(f"initial performance ===============")
    eval(model, x_train, y_train)

    for epoch in range(epochs):
        for i in range(0, n, bs):
            s = slice(i, min(n,i+bs))
            xb,yb = x_train[s],y_train[s]
            preds = model(xb)
            loss = loss_func(preds, yb)
            loss.backward()
            with torch.no_grad():
                for l in model.layers:
                    if hasattr(l, 'weight'):
                        l.weight -= l.weight.grad * lr
                        l.bias   -= l.bias.grad   * lr
                        l.weight.grad.zero_()
                        l.bias  .grad.zero_()
        print(f"end of epoch {epoch} performance ===============")
        eval(model, x_train, y_train)

fit_0(model, x_train, y_train, bs)

# Refactor the trainig loop

## Using `nn.Module`

In [ ]:
# m1 = nn.Module()
# m1.foo = nn.Linear(3,4)
# print(m1)
# print("================")
# print(m1.named_children())
# print("================")
# print(list(m1.named_children()))
# print("================")
# print(list(m1.parameters()))

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_in, nh, n_out):
        super().__init__()
        self.l1 = nn.Linear(n_in,nh)
        self.l2 = nn.Linear(nh,n_out)
        self.relu = nn.ReLU()
        
    def forward(self, x): return self.l2(self.relu(self.l1(x)))

In [ ]:
model = MLP(m, nh, 10)
print(model)
print("================")
for name,l in model.named_children(): print(f"{name}: {l}")
print("================")
for p in model.parameters(): print(p.shape)

In [ ]:
def fit_1(model, x_train, y_train, bs, lr = 0.5, epochs = 3):
    print(f"initial performance ===============")
    eval(model, x_train, y_train)

    for epoch in range(epochs):
        for i in range(0, n, bs):
            s = slice(i, min(n,i+bs))
            xb,yb = x_train[s],y_train[s]
            preds = model(xb)
            loss = loss_func(preds, yb)
            loss.backward()
            with torch.no_grad():
                for p in model.parameters(): p -= p.grad * lr
                model.zero_grad()
        print(f"end of epoch {epoch} performance ===============")
        eval(model, x_train, y_train)

fit_1(model, x_train, y_train, bs)

### Re-construct `nn.Module`

Behind the scenes, PyTorch overrides the `__setattr__` function in `nn.Module` so that the submodules you define are properly registered as parameters of the model.

In [ ]:
class MyModule:
    def __init__(self, n_in, nh, n_out):
        self._modules = {}
        self.l1 = nn.Linear(n_in,nh)
        self.l2 = nn.Linear(nh,n_out)

    def __setattr__(self,k,v):
        if not k.startswith("_"): self._modules[k] = v
        super().__setattr__(k,v)

    def __repr__(self): return f'{self._modules}'
    
    def parameters(self):
        for l in self._modules.values(): yield from l.parameters()

mdl = MyModule(m,nh,10)
print("================")
print(mdl)
print("================")
for p in mdl.parameters(): print(p.shape)

❓ Why do we use `super().__setattr__` inside the `__setattr__` method?
- **Avoiding Infinite Recursion**: When you use `self.attribute = value` within an object's custom `__setattr__` method, Python interprets this assignment as another call to the _same_ `__setattr__` method. This creates an infinite recursion loop, quickly leading to a `RecursionError`.
- **Bypassing Custom Logic**: Calling `super().__setattr__(name, value)` (or `object.__setattr__(self, name, value)` in some cases) explicitly delegates the attribute assignment to the parent class's implementation (ultimately the default `object` class's implementation). This bypasses your custom `__setattr__` logic, allowing the attribute to be set in the object's internal dictionary (`__dict__`) in the standard way.

## Registering modules

### User-defined

In [ ]:
from functools import reduce
layers = [nn.Linear(m,nh), nn.ReLU(), nn.Linear(nh,10)]

class Model(nn.Module):
    def __init__(self, layers):
        super().__init__()
        self.layers = layers
        for i,l in enumerate(self.layers): self.add_module(f'layer_{i}', l)

    def forward(self, x): return reduce(lambda val,layer: layer(val), self.layers, x)

model = Model(layers)
print("================")
print(model)
print("================")
print(model(xb).shape)


❓How does `functools.reduce` work?
- [functools higher-order function](https://docs.python.org/3/library/functools.html#functools.reduce) including `reduce`
- [itertools higher-order functions](https://docs.python.org/3/library/itertools.html#itertools.accumulate) including `accumulate`

### `nn.ModuleList`

In [ ]:
class SequentialModel(nn.Module):
    def __init__(self, layers):
        super().__init__()
        self.layers = nn.ModuleList(layers)
        
    def forward(self, x):
        for l in self.layers: x = l(x)
        return x

model = SequentialModel(layers)
print(model)

In [ ]:
fit_1(model, x_train, y_train, bs)

### `nn.Sequential`

In [ ]:
model = nn.Sequential(nn.Linear(m,nh), nn.ReLU(), nn.Linear(nh,10))
print("================")
print(model)
print("================")
fit_1(model, x_train, y_train, bs)

## Adding optimizer

### User-defined

In [ ]:
class Optimizer:
    def __init__(self, params, lr): self.params,self.lr=list(params),lr

    def step(self):
        with torch.no_grad():
            for p in self.params: p -= p.grad * self.lr

    def zero_grad(self):
        for p in self.params: p.grad.data.zero_()

In [ ]:
model = nn.Sequential(nn.Linear(m,nh), nn.ReLU(), nn.Linear(nh,10))
opt = Optimizer(model.parameters(), lr=0.5)

def fit_2(model, x_train, y_train, bs, opt, epochs = 3):
    print(f"initial performance ===============")
    eval(model, x_train, y_train)
    for epoch in range(epochs):
        for i in range(0, n, bs):
            s = slice(i, min(n,i+bs))
            xb,yb = x_train[s],y_train[s]
            preds = model(xb)
            loss = loss_func(preds, yb)
            loss.backward()
            opt.step()
            opt.zero_grad()
        print(f"end of epoch {epoch} performance ===============")
        eval(model, x_train, y_train)

fit_2(model, x_train, y_train, bs, opt)

### Using Pytorch optimizer

- PyTorch already provides this exact functionality in `optim.SGD`
- (it also handles stuff like **momentum**, which we'll look at later)

In [ ]:
from torch import optim
model = nn.Sequential(nn.Linear(m,nh), nn.ReLU(), nn.Linear(nh,10))
opt = optim.SGD(model.parameters(), lr=0.5)
fit_2(model, x_train, y_train, bs, opt)

## Adding `Dataset` and `DataLoader`

### Dataset

In [ ]:
class Dataset:
    def __init__(self, x, y): self.x,self.y = x,y
    def __len__(self): return len(self.x)
    def __getitem__(self, i): return self.x[i],self.y[i]

train_ds,valid_ds = Dataset(x_train, y_train),Dataset(x_valid, y_valid)
assert len(train_ds)==len(x_train)
assert len(valid_ds)==len(x_valid)

xb,yb = train_ds[0:5]
assert xb.shape==(5,28*28)
assert yb.shape==(5,)
print(xb,yb)

In [ ]:
def eval_ds(model, valid_ds, bs):
    pred_list = []
    label_list = []
    with torch.no_grad():
        for i in range(0, n, bs):
            xb,yb = valid_ds[i:min(n,i+bs)]
            pred_list.append(model(xb))
            label_list.append(yb)
        preds = torch.cat(pred_list, dim=0)
        labels = torch.cat(label_list, dim=0)
        loss = loss_func(preds, labels)
        report(loss, preds, labels)

def fit_3(model, train_ds, valid_ds, bs, opt, epochs = 3):
    print(f"initial performance ===============")
    eval_ds(model, valid_ds, bs)
    for epoch in range(epochs):
        for i in range(0, n, bs):
            xb,yb = train_ds[i:min(n,i+bs)]
            preds = model(xb)
            loss = loss_func(preds, yb)
            loss.backward()
            opt.step()
            opt.zero_grad()
        print(f"end of epoch {epoch} performance ===============")
        eval_ds(model, valid_ds, bs)

In [ ]:
model = nn.Sequential(nn.Linear(m,nh), nn.ReLU(), nn.Linear(nh,10))
opt = optim.SGD(model.parameters(), lr=0.5)
fit_3(model, train_ds, valid_ds, bs, opt)

### DataLoader

In [ ]:
class DataLoader:
    def __init__(self, ds, bs): self.ds,self.bs = ds,bs
    def __iter__(self):
        for i in range(0, len(self.ds), self.bs): yield self.ds[i:i+self.bs]

train_dl = DataLoader(train_ds, bs)
valid_dl = DataLoader(valid_ds, bs)

xb,yb = next(iter(valid_dl))
print(xb.shape)
print("====================")
print(yb)
print("====================")
plt.figure(figsize=(3,3))
plt.imshow(xb[0].view(28,28))
plt.title(f"true label: {yb[0]}");

In [ ]:
def eval_dl(model, valid_dl):
    pred_list = []
    label_list = []
    with torch.no_grad():
        for xb,yb in valid_dl:
            pred_list.append(model(xb))
            label_list.append(yb)
        preds = torch.cat(pred_list, dim=0)
        labels = torch.cat(label_list, dim=0)
        loss = loss_func(preds, labels)
        report(loss, preds, labels)

def fit_4(model, train_dl, valid_dl, opt, epochs = 3):
    print(f"initial performance ===============")
    eval_dl(model, valid_dl)
    for epoch in range(epochs):
        for xb,yb in train_dl:
            preds = model(xb)
            loss = loss_func(preds, yb)
            loss.backward()
            opt.step()
            opt.zero_grad()
        print(f"end of epoch {epoch} performance ===============")
        eval_dl(model, valid_dl)


model = nn.Sequential(nn.Linear(m,nh), nn.ReLU(), nn.Linear(nh,10))
opt = optim.SGD(model.parameters(), lr=0.5)
fit_4(model, train_dl, valid_dl, opt)